# ChaCha20 auf dem CW-Lite (STM32F3): Test & erste Traces

Dieses Notebook testet die Firmware `firmware/mcu/simpleserial-chacha20`, also ChaCha20 in der **IETF-Variante** (RFC 8439: 96-Bit-Nonce, 32-Bit-Counter).

**Ablauf:**
1. Setup: Scope und Target verbinden
2. Firmware bauen und flashen
3. Referenz (pycryptodome) gegen die RFC-8439-Testvektoren prüfen
4. Target gegen die Referenz validieren
5. Trace-Länge bestimmen (passt der Block ins Sample-Fenster?)
6. Traces aufnehmen und speichern (fester Schlüssel, zufällige Nonce und zufälliger Counter, Grundlage für eine spätere CPA)

**SimpleSerial-Befehle der Firmware:**

| Befehl | Länge | Funktion |
|---|---|---|
| `'k'` | 32 B | Schlüssel setzen |
| `'n'` | 16 B | Counter (4 B, little endian) und Nonce (12 B) |
| `'p'` | 16 B | 16 B Klartext verschlüsseln, Antwort `'r'` mit 16 B |
| `'x'` | 0 B | Schlüssel, Nonce und Counter auf null setzen |

## 1. Setup

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM  = 'CWLITEARM'    #STM32
SS_VER    = 'SS_VER_1_1'

FW_DIR = "../../firmware/mcu/simpleserial-chacha20"

%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24445                     to 5000                     
scope.adc.trig_count                     changed from 27959187                  to 41697067                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 20763610                  to 29542304                 
scope.clock.adc_rate                     changed from 20763610.0                to 29542304.0        

## 2. Firmware bauen und flashen

In [2]:
%%bash -s "$FW_DIR"
cd "$1"
make -j

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
arm-none-eabi-gcc (Arm GNU Toolchain 15.3.Rel1 (Build arm-15.149)) 15.3.1 20260627
Copyright (C) 2025 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
Compiling:
.
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Assembling: .././hal//stm32f3/stm32f3_startup.S
-en     .././simpleserial/simpleserial.c ...
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I.././simpleserial/ -I.././hal/ -I.././hal/ -I.././hal//stm32f3 -I.././hal//stm32f3/CMSIS -I.././hal//stm32f3/CMSIS/core -I.././hal//stm32f3/CMSIS/device -I.././hal//stm32f4/Legacy -I.././simp

/Applications/ArmGNUToolchain/15.3.rel1/arm-none-eabi/bin/../lib/gcc/arm-none-eabi/15.3.1/../../../../arm-none-eabi/bin/ld: warning: simpleserial-chacha20-CWLITEARM.elf has a LOAD segment with RWX permissions


Memory region         Used Size  Region Size  %age Used
             RAM:        1736 B        40 KB      4.24%
             ROM:        5344 B       256 KB      2.04%
-e Done!
.
.
.
.
.
Creating load file for EEPROM: simpleserial-chacha20-CWLITEARM.eep
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
	--change-section-lma .eeprom=0 --no-change-warnings -O ihex simpleserial-chacha20-CWLITEARM.elf simpleserial-chacha20-CWLITEARM.eep || exit 0
Creating load file for Flash: simpleserial-chacha20-CWLITEARM.bin
Creating load file for Flash: simpleserial-chacha20-CWLITEARM.hex
Creating Extended Listing: simpleserial-chacha20-CWLITEARM.lss
arm-none-eabi-objdump -h -S -z simpleserial-chacha20-CWLITEARM.elf > simpleserial-chacha20-CWLITEARM.lss
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature simpleserial-chacha20-CWLITEARM.elf simpleserial-chacha20-CWLITEARM.hex
Creating Symbol Table: simpleserial-chacha20-CWLITEARM.sym
arm-none-eabi-nm -n sim

In [3]:
cw.program_target(scope, prog, "{}/simpleserial-chacha20-{}.hex".format(FW_DIR, PLATFORM))
reset_target(scope)
target.flush()

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 5343 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 5343 bytes


## 3. Referenzimplementierungen prüfen

Es gibt zwei Referenzen:

- **`pycryptodome`**: unabhängige, etablierte Implementierung. Sie wählt die Variante anhand der Nonce-Länge, 12 Byte entsprechen IETF/RFC 8439. Falls der Import fehlschlägt, einmal `%pip install pycryptodome` ausführen und den Kernel neu starten.
- **`chacha20_sim.py`**: 1:1-Python-Nachbau der Firmware-Datei `chacha20.c` mit gleichen Funktionen, Schleifen und Reihenfolge. Damit lassen sich später auch Zwischenwerte der Runden für die CPA berechnen.

Beide werden gegen den RFC-8439-Testvektor und gegeneinander geprüft.

In [4]:
import os
import numpy as np
from Crypto.Cipher import ChaCha20
from chacha20_sim import ChaCha20XOR as sim_ChaCha20XOR
from chacha20_sim import HW

def chacha_ref(key, ctr, nonce, pt):
    # ChaCha20 (IETF) Referenz (pycryptodome): Keystream ab Block 'ctr' XOR pt
    assert len(key) == 32 and len(nonce) == 12
    c = ChaCha20.new(key=bytes(key), nonce=bytes(nonce))
    c.seek(64 * ctr)
    return c.encrypt(bytes(pt))

def rand_ctr():
    # Zufälliger 32-Bit-Counter. 0xFFFFFFFF wird ausgelassen, weil pycryptodome
    # (IETF) für den letzten Block keinen seek() erlaubt; Firmware und
    # Simulation können ihn, das ist nur eine Einschränkung der Referenz.
    while True:
        ctr = int.from_bytes(os.urandom(4), 'little')
        if ctr != 0xFFFFFFFF:
            return ctr

def chacha_sim(key, ctr, nonce, pt, trace=None):
    # ChaCha20 Simulation der Firmware (chacha20_sim.py), Aufruf wie in C.
    # Optional: Liste 'trace' übergeben, sie wird mit den HW-Werten gefüllt.
    if trace is None:
        trace = []
    out = [0] * len(pt)
    sim_ChaCha20XOR(key, ctr, nonce, pt, out, len(pt), trace)
    return bytes(out)

# RFC 8439, Abschnitt 2.4.2 (Testvektor für die Verschlüsselung)
RFC_KEY   = bytes(range(32))
RFC_NONCE = bytes.fromhex("000000000000004a00000000")
RFC_CTR   = 1
RFC_PT    = b"Ladies and Gentlemen of the class of '99: If I could offer you only one tip for the future, sunscreen would be it."
RFC_CT    = bytes.fromhex(
    "6e2e359a2568f98041ba0728dd0d6981e97e7aec1d4360c20a27afccfd9fae0b"
    "f91b65c5524733ab8f593dabcd62b3571639d624e65152ab8f530c359f0861d8")  # erste 64 B

assert chacha_ref(RFC_KEY, RFC_CTR, RFC_NONCE, RFC_PT)[:64] == RFC_CT
print("pycryptodome stimmt mit RFC 8439 überein ✔")
assert chacha_sim(RFC_KEY, RFC_CTR, RFC_NONCE, RFC_PT)[:64] == RFC_CT
print("Simulation   stimmt mit RFC 8439 überein ✔")

# Simulation vs. pycryptodome mit Zufallsvektoren
N_CROSS = 100
for i in range(N_CROSS):
    key, nonce = os.urandom(32), os.urandom(12)
    ctr = rand_ctr()
    pt  = os.urandom(16)
    assert chacha_sim(key, ctr, nonce, pt) == chacha_ref(key, ctr, nonce, pt), f"Mismatch bei Vektor {i}"
print(f"Simulation   stimmt bei {N_CROSS} Zufallsvektoren mit pycryptodome überein ✔")

pycryptodome stimmt mit RFC 8439 überein ✔


TypeError: ChaCha20XOR() takes 6 positional arguments but 7 were given

### Simulierte Hamming Weights

Die Simulation zeichnet nach jeder Zuweisung an eine ChaCha20-Variable `HW(neuer Wert)` auf. So sieht der Leakage-Trace des Hamming-Weight-Modells für einen 16-Byte-Aufruf aus. Die farbigen Bereiche entsprechen den Abschnitten von `ChaCha20XOR()`, die gestrichelten Linien trennen die 10 Doppelrunden.

In [ ]:
import matplotlib.pyplot as plt

hw_trace = []
chacha_sim(RFC_KEY, RFC_CTR, RFC_NONCE, RFC_PT[:16], hw_trace)

# Abschnitte des Traces (Indexbereiche für inlen = 16)
sections = [
    (0,   60,  "Init State",      "tab:blue"),
    (60,  76,  "memcpy",          "tab:gray"),
    (76,  716, "20 Runden",       "tab:orange"),
    (716, 732, "x[i] += in[i]",   "tab:green"),
    (732, 796, "Serialize",       "tab:purple"),
    (796, 797, "s[12]++",         "tab:brown"),
    (797, len(hw_trace), "XOR out", "tab:red"),
]

fig, ax = plt.subplots(figsize=(15, 4.5))
for start, end, label, color in sections:
    ax.axvspan(start - 0.5, end - 0.5, color=color, alpha=0.12, label=label)
for k in range(1, 10):                       # Grenzen der 10 Doppelrunden (je 8 QR x 8 Werte)
    ax.axvline(76 + 64 * k - 0.5, color="tab:orange", ls=":", lw=0.8)
ax.step(range(len(hw_trace)), hw_trace, where="mid", lw=0.8, color="black")
ax.set_xlim(-0.5, len(hw_trace) - 0.5)
ax.set_ylim(0, 33)
ax.set_xlabel("Index im Trace (Zuweisung)")
ax.set_ylabel("Hamming Weight")
ax.set_title(f"Simulierter HW-Leakage-Trace von ChaCha20XOR (16 Byte, {len(hw_trace)} Werte)")
ax.legend(loc="upper center", ncol=len(sections), fontsize=8, bbox_to_anchor=(0.5, -0.15))
plt.tight_layout()
plt.show()

## 4. Target validieren

In [ ]:
def target_encrypt(key, ctr, nonce, pt):
    # Setzt Key/Counter/Nonce auf dem Target und verschlüsselt 16 Byte
    target.simpleserial_write('k', bytearray(key))
    if target.simpleserial_wait_ack() is None:
        raise IOError("kein ACK auf 'k'")
    target.simpleserial_write('n', bytearray(ctr.to_bytes(4, 'little') + bytes(nonce)))
    if target.simpleserial_wait_ack() is None:
        raise IOError("kein ACK auf 'n'")
    target.simpleserial_write('p', bytearray(pt))
    ct = target.simpleserial_read('r', 16)
    if ct is None:
        raise IOError("keine Antwort auf 'p'")
    return bytes(ct)

# 1) RFC-Testvektor (die ersten 16 Byte)
ct = target_encrypt(RFC_KEY, RFC_CTR, RFC_NONCE, RFC_PT[:16])
print("RFC  target:", ct.hex())
print("RFC  ref   :", RFC_CT[:16].hex())
print("RFC  sim   :", chacha_sim(RFC_KEY, RFC_CTR, RFC_NONCE, RFC_PT[:16]).hex())
assert ct == RFC_CT[:16], "Target liefert beim RFC-Vektor ein falsches Ergebnis!"
assert ct == chacha_sim(RFC_KEY, RFC_CTR, RFC_NONCE, RFC_PT[:16]), "Target weicht beim RFC-Vektor von der Simulation ab!"

# 2) Zufallsvektoren: Target vs. pycryptodome und vs. Simulation
N_VALIDATE = 20
for i in range(N_VALIDATE):
    key, nonce, pt = os.urandom(32), os.urandom(12), os.urandom(16)
    ctr = rand_ctr()
    ct  = target_encrypt(key, ctr, nonce, pt)
    assert ct == chacha_ref(key, ctr, nonce, pt), f"Target != pycryptodome bei Vektor {i}"
    assert ct == chacha_sim(key, ctr, nonce, pt), f"Target != Simulation bei Vektor {i}"

print(f"Target stimmt bei RFC-Vektor + {N_VALIDATE} Zufallsvektoren mit pycryptodome und Simulation überein ✔")

## 5. Trace-Länge bestimmen

`scope.adc.trig_count` zählt, wie viele ADC-Samples der Trigger high war, also wie lange `ChaCha20XOR()` läuft. Der CW-Lite kann maximal etwa 24 400 Samples pro Trace speichern.

In [ ]:
key, nonce, pt = os.urandom(32), os.urandom(12), os.urandom(16)
ctr = 0
target.simpleserial_write('k', bytearray(key));  target.simpleserial_wait_ack()
target.simpleserial_write('n', bytearray(ctr.to_bytes(4, 'little') + nonce));  target.simpleserial_wait_ack()

scope.adc.samples = 24400
scope.arm()
target.simpleserial_write('p', bytearray(pt))
if scope.capture():
    print("Timeout!")
target.simpleserial_read('r', 16)

trig_samples = scope.adc.trig_count
print("Trigger high für     :", trig_samples, "Samples")
print("ADC-Takt / Target-Takt:", scope.clock.adc_src)
print("Max. Samples (CW-Lite):", 24400)
if trig_samples > 24400:
    print("→ Der ganze Block passt nicht ins Fenster: scope.adc.offset nutzen oder nur die ersten Runden aufnehmen.")
else:
    print("→ Der ganze Block passt ins Fenster.")

Ein erster Trace zur Orientierung. Die 10 Doppelrunden sollten als sich wiederholendes Muster erkennbar sein:

In [ ]:
import matplotlib.pyplot as plt

trace = scope.get_last_trace()
plt.figure(figsize=(14, 4))
plt.plot(trace, lw=0.5)
plt.axvline(min(trig_samples, len(trace)), color='r', ls='--', label='Trigger low')
plt.xlabel("Sample"); plt.ylabel("Leistung (normiert)")
plt.title("ChaCha20 auf STM32F3, ein Trace")
plt.legend(); plt.show()

## 6. Traces aufnehmen

Fester (geheimer) Schlüssel, für jeden Trace zufälliger Counter und zufällige Nonce. Das entspricht dem Angreifermodell für eine CPA auf die ersten Quarterrounds: Die Wörter 12–15 sind bekannt und variieren. Jede Antwort wird gegen pycryptodome **und** gegen die Simulation geprüft. Nur Traces mit korrekter Antwort werden gespeichert.

In [ ]:
from tqdm.notebook import trange

N       = 1000
SAMPLES = 5000        # nach Abschnitt 5 anpassen (max. 24400)
OFFSET  = 0           # Samples nach dem Trigger überspringen

scope.adc.samples = SAMPLES
scope.adc.offset  = OFFSET

key = os.urandom(32)  # fester Schlüssel: das Ziel der späteren CPA
target.simpleserial_write('k', bytearray(key)); target.simpleserial_wait_ack()

traces, ctrnonce_in, pt_in, ct_out = [], [], [], []
errors = 0          # Timeouts / keine Antwort
mismatch_ref = 0    # Antwort != pycryptodome
mismatch_sim = 0    # Antwort != Simulation

for i in trange(N, desc="Capturing traces"):
    nonce = os.urandom(12)
    ctr   = rand_ctr()
    pt    = os.urandom(16)

    target.simpleserial_write('n', bytearray(ctr.to_bytes(4, 'little') + nonce))
    target.simpleserial_wait_ack()

    scope.arm()
    target.simpleserial_write('p', bytearray(pt))
    if scope.capture():
        print("Target timed out!"); errors += 1
        continue
    ct = target.simpleserial_read('r', 16)
    if ct is None:
        errors += 1
        continue
    ok_ref = bytes(ct) == chacha_ref(key, ctr, nonce, pt)
    ok_sim = bytes(ct) == chacha_sim(key, ctr, nonce, pt)
    mismatch_ref += not ok_ref
    mismatch_sim += not ok_sim
    if not (ok_ref and ok_sim):
        continue

    traces.append(scope.get_last_trace())
    ctrnonce_in.append(np.frombuffer(ctr.to_bytes(4, 'little') + nonce, dtype=np.uint8))
    pt_in.append(np.frombuffer(pt, dtype=np.uint8))
    ct_out.append(np.frombuffer(bytes(ct), dtype=np.uint8))

traces      = np.array(traces)
ctrnonce_in = np.array(ctrnonce_in)
pt_in       = np.array(pt_in)
ct_out      = np.array(ct_out)
print(f"{len(traces)} gültige Traces, {errors} Timeouts, "
      f"{mismatch_ref} Abweichungen zu pycryptodome, {mismatch_sim} Abweichungen zur Simulation")

In [ ]:
np.savez("chacha20_traces.npz",
         traces=traces, ctrnonce=ctrnonce_in, pt=pt_in, ct=ct_out,
         key=np.frombuffer(key, dtype=np.uint8))
print("gespeichert: chacha20_traces.npz")

In [ ]:
plt.figure(figsize=(14, 4))
for t in traces[:5]:
    plt.plot(t, lw=0.5)
plt.xlabel("Sample"); plt.title("Erste 5 Traces")
plt.show()

## Hardware trennen

In [ ]:
scope.dis()
target.dis()

In [ ]:


def mean(X):
    return np.sum(X, axis=0) / X.shape[0]

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X - X_bar) ** 2, axis=0) / X.shape[0])

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X - X_bar) * (Y - Y_bar), axis=0) / X.shape[0]

In [ ]:
maxcpa = [0] * 256
textin_array = []

# we don't need to redo the mean and std dev calculations
# for each key guess
t_bar = mean(traces)
o_t = std_dev(traces, t_bar)
print(HW(chacha_sim(key, ctr, nonce, pt)))
for kguess in range(0, 256):
    hws = np.array([[HW(chacha_sim(key, ctr, nonce, pt)) for textin in textin_array]]).transpose()
    
    # hws changes with every guess, so its mean and std dev go inside the loop
    hws_bar = mean(hws)
    o_hws = std_dev(hws, hws_bar)

    #correlation = cov(traces, t_bar, hws, hws_bar)
    #cpaoutput = correlation / (o_t * o_hws)

    #maxcpa[kguess] = max(abs(cpaoutput))

#guess = np.argmax(maxcpa)
#guess_corr = max(maxcpa)

#print("Key guess: ", hex(guess))
#print("Correlation: ", guess_corr)

## 7. CPA-Angriff auf ChaCha20 (offline)

Angegriffen wird die **erste (Spalten-)Runde**. Pro Spalte `j` (0..3) rechnet
`QR(j, 4+j, 8+j, 12+j)`. Das variierende, *bekannte* Wort ist `d = s[12+j]`
(Counter bzw. Nonce-Wörter), die Konstante ist `s[j]`, die geheimen Key-Wörter
sind `s[4+j]` (oben) und `s[8+j]` (unten).

- **Stufe 1 (Op 2):** `x[d] = rotl(x[d] ^ (const + s[4+j]), 16)`.
  HW ist rotationsinvariant → Leckage ist `HW(d ^ W)` mit `W = (const + s[4+j]) mod 2**32`.
  Reines XOR → jedes Bit von `W` folgt aus dem **Vorzeichen** von `corr(trace, bit_i(d))`.
- **Stufe 2 (Op 3):** `x[c] += x[d]` → Leckage `HW(s[8+j] + d2)`, mit `d2` = Ergebnis aus Stufe 1.
  Addition mit Carry → bitweise LSB→MSB, Carry aus den schon bekannten Bits.

Der Chip leakt die HW des **ganzen 32-bit-Worts** (Register werden pro Zyklus genullt
→ HW-Modell). Darum ist das Bit-/Vorzeichen-Verfahren hier robuster als die klassische
256-Hypothesen-pro-Byte-CPA (die nur 8 von 32 Bit modelliert → schlechtes SNR).

PoI und Leckage-Vorzeichen werden einmal mit dem bekannten Testschlüssel bestimmt
(*Profiling*); der eigentliche Recovery-Schritt nutzt den Key danach nicht mehr.
Diese Sektion lädt aus `chacha20_traces.npz` und läuft unabhängig vom Capture oben.

### 7.0 Daten & Hilfsfunktionen

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d   = np.load("chacha20_traces.npz", allow_pickle=True)
T   = d["traces"].astype(np.float64)       # (N, S)
cn  = d["ctrnonce"]                         # (N,16) = counter(4) || nonce(12)
pt_ = d["pt"]; ct_ = d["ct"]
KEY = bytes(d["key"].tolist())              # nur fuer Profiling / Erfolgskontrolle
N, S = T.shape
print(f"{N} Traces x {S} Samples")

MASK  = 0xffffffff
CONST = [0x61707865, 0x3320646e, 0x79622d32, 0x6b206574]

# bekannte, variierende Eingabe-Woerter je Spalte: s[12+j]
#   j=0 -> Counter (cn[:,0:4]),  j=1..3 -> Nonce-Woerter (cn[:,4:8] / 8:12 / 12:16)
in_word = np.array([[int.from_bytes(bytes(cn[i, 4*w:4*w+4].tolist()), "little")
                     for w in range(4)] for i in range(N)], dtype=np.uint64)
# Ground-Truth-Key-Woerter s[4]..s[11] (Index j=0..7) -- nur zur Kontrolle
key_word_true = [int.from_bytes(KEY[4*j:4*j+4], "little") for j in range(8)]

def hw(v):
    v = np.asarray(v, dtype=np.uint64); c = np.zeros(v.shape, dtype=np.uint64)
    for i in range(32):
        c += (v >> np.uint64(i)) & np.uint64(1)
    return c.astype(np.float64)

def rotl32(x, n):
    x = int(x) & MASK
    return ((x << n) | (x >> (32 - n))) & MASK

# vorberechnete, zentrierte Traces -> schnelle spaltenweise Korrelation
_Tc   = T - T.mean(axis=0)
_Tden = np.sqrt((_Tc**2).sum(axis=0))
def corr_all(model):                        # model:(N,) -> r:(S,)
    m = model - model.mean()
    return (m @ _Tc) / (np.sqrt((m**2).sum()) * _Tden + 1e-12)

### 7.1 Point-of-Interest & Leckage-Profiling (bekannter Key)

Volles HW-Modell beider Ziel-Operationen mit dem bekannten Testschlüssel. Daraus
PoI (Sample mit maximalem |r|) **und** das Vorzeichen des Leckage-Modells je Spalte.

In [ ]:
poi_op2, poi_op3, alpha_op2, alpha_op3 = [], [], [], []
for j in range(4):
    dw = in_word[:, j]
    W  = (CONST[j] + key_word_true[j]) & MASK
    c2 = corr_all(hw(dw ^ np.uint64(W)))
    p2 = int(np.argmax(np.abs(c2))); poi_op2.append(p2); alpha_op2.append(np.sign(c2[p2]))
    d2 = np.array([rotl32(int(dw[i]) ^ W, 16) for i in range(N)], dtype=np.uint64)
    c3 = corr_all(hw((d2 + np.uint64(key_word_true[4 + j])) & np.uint64(MASK)))
    p3 = int(np.argmax(np.abs(c3))); poi_op3.append(p3); alpha_op3.append(np.sign(c3[p3]))
    print(f"Spalte {j}:  Op2 PoI={p2:4d} |r|={abs(c2[p2]):.2f}   "
          f"Op3 PoI={p3:4d} |r|={abs(c3[p3]):.2f}")

dw = in_word[:, 0]
c0 = corr_all(hw(dw ^ np.uint64((CONST[0] + key_word_true[0]) & MASK)))
plt.figure(figsize=(12, 3.5))
plt.plot(np.abs(c0), lw=0.7)
plt.axvline(poi_op2[0], color="r", ls="--",
            label=f"PoI {poi_op2[0]} (|r|={abs(c0[poi_op2[0]]):.2f})")
plt.title("Profiling Spalte 0, Op 2: |corr| ueber Samples")
plt.xlabel("Sample"); plt.ylabel("|r|"); plt.legend(); plt.show()

### 7.2 Stufe 1 — obere Key-Wörter `s[4..7]` (XOR, Bit-Vorzeichen)

`W = const + s[4+j]` folgt bitweise aus dem Vorzeichen von `corr(trace, bit_i(d))`;
`s[4+j] = (W - const) mod 2**32`. Kein Key im Recovery-Schritt.

In [ ]:
def recover_xor(dw, poi, alpha):
    """Leckage HW(dw ^ W): jedes Bit aus dem Vorzeichen von corr(trace, bit_i(dw))."""
    W = 0
    for i in range(32):
        bit = ((dw >> np.uint64(i)) & np.uint64(1)).astype(np.float64)
        if np.sign(corr_all(bit)[poi]) != alpha:
            W |= 1 << i
    return W

key_word = [0] * 8
for j in range(4):
    W = recover_xor(in_word[:, j], poi_op2[j], alpha_op2[j])
    key_word[j] = (W - CONST[j]) & MASK
    ok = "OK" if key_word[j] == key_word_true[j] else "FAIL"
    print(f"s[{4+j}] = 0x{key_word[j]:08x}  (wahr 0x{key_word_true[j]:08x})  {ok}")

### 7.3 Stufe 2 — untere Key-Wörter `s[8..11]` (ADD mit Carry)

Summen-Bit `i` ist `a_i ^ k_i` mit `k_i = b_i ^ carry_i` (bekannt, sobald die
niedrigeren `a`-Bits feststehen). Damit dasselbe Vorzeichen-Verfahren LSB→MSB.

> **Hinweis / ehrliches Limit:** Die ADD-Stufe ist deutlich schwächer als die
> XOR-Stufe. Der Chip leakt die HW des *ganzen* Worts, und die Carry-Kette koppelt
> die Bits, wodurch das Signal pro Bit verschmiert. Bei **N = 1000** werden typisch
> nur ~2/4 der unteren Wörter voll korrekt. Für die vollen 256 bit mehr Traces
> aufnehmen (Capture-Zelle, `N` erhöhen — der Konvergenz-Plot unten zeigt den Bedarf).

In [ ]:
def recover_add(b_known, poi, alpha):
    """Leckage HW(a + b_known): a bitweise LSB->MSB, Carry aus bekannten Bits."""
    a = 0
    carry = np.zeros(N, dtype=np.uint64)
    for i in range(32):
        bi = (b_known >> np.uint64(i)) & np.uint64(1)
        ki = (bi ^ carry).astype(np.float64)            # Summen-Bit i = a_i ^ k_i
        ai = 1 if np.sign(corr_all(ki)[poi]) != alpha else 0
        a |= ai << i
        av = np.uint64(ai)
        carry = ((av & bi) | (av & carry) | (bi & carry)).astype(np.uint64)
    return a

for j in range(4):
    dw = in_word[:, j]
    W  = (CONST[j] + key_word[j]) & MASK                 # aus Stufe 1 rekonstruiert
    d2 = np.array([rotl32(int(dw[i]) ^ W, 16) for i in range(N)], dtype=np.uint64)
    key_word[4 + j] = recover_add(d2, poi_op3[j], alpha_op3[j])
    ok = "OK" if key_word[4 + j] == key_word_true[4 + j] else "FAIL"
    print(f"s[{8+j}] = 0x{key_word[4+j]:08x}  (wahr 0x{key_word_true[4+j]:08x})  {ok}")

### 7.4 Schlüssel zusammensetzen & verifizieren

In [ ]:
rec_key = b"".join(int(key_word[j]).to_bytes(4, "little") for j in range(8))
print("rekonstruiert:", rec_key.hex())
print("wahr         :", KEY.hex())

nb_all = sum(a == b for a, b in zip(rec_key, KEY))
nb_up  = sum(a == b for a, b in zip(rec_key[:16], KEY[:16]))
nb_lo  = sum(a == b for a, b in zip(rec_key[16:], KEY[16:]))
print(f"korrekte Key-Bytes: {nb_all}/32  (obere 16: {nb_up}/16, untere 16: {nb_lo}/16)")

# unabhaengige Verifikation ueber ein gespeichertes (pt, ct)-Paar
from Crypto.Cipher import ChaCha20
i   = 0
ctr = int.from_bytes(bytes(cn[i, :4].tolist()), "little")
non = bytes(cn[i, 4:].tolist())
c   = ChaCha20.new(key=rec_key, nonce=non); c.seek(64 * ctr)
print("Re-Encrypt-Check gegen gespeichertes ct:", c.encrypt(bytes(pt_[i])) == bytes(ct_[i]))

### 7.5 Konvergenz — wie viele Traces braucht es?

|r| des korrekten Voll-Wort-Modells am PoI über der Trace-Zahl, für XOR (Op 2) und
ADD (Op 3) der Spalte 0. Zeigt, warum die ADD-Stufe mehr Traces verlangt.

In [ ]:
Ns = np.unique(np.linspace(100, N, 15).astype(int))
dw = in_word[:, 0]
W0 = (CONST[0] + key_word_true[0]) & MASK
d2 = np.array([rotl32(int(dw[i]) ^ W0, 16) for i in range(N)], dtype=np.uint64)
H2 = hw(dw ^ np.uint64(W0))
H3 = hw((d2 + np.uint64(key_word_true[4])) & np.uint64(MASK))

def r_at(H, t, n):
    h  = H[:n] - H[:n].mean()
    tt = T[:n, t] - T[:n, t].mean()
    return abs((h @ tt) / (np.sqrt((h**2).sum()) * np.sqrt((tt**2).sum()) + 1e-12))

r2 = [r_at(H2, poi_op2[0], n) for n in Ns]
r3 = [r_at(H3, poi_op3[0], n) for n in Ns]
plt.figure(figsize=(10, 3.5))
plt.plot(Ns, r2, "o-", label="Op 2 (XOR, oberes Wort)")
plt.plot(Ns, r3, "s-", label="Op 3 (ADD, unteres Wort)")
plt.xlabel("Anzahl Traces N"); plt.ylabel("|r| am PoI (Spalte 0)")
plt.title("Konvergenz des Voll-Wort-Leckagemodells")
plt.legend(); plt.grid(alpha=0.3); plt.show()